# 03 · Is the Kaggle test set different from train? (adversarial validation)

The LightGBM baseline scored **0.7585 in CV** but **0.7451 on Kaggle (private)**, a gap of ~0.013,
about 2.5× the fold-to-fold standard deviation. Leakage is unlikely (LightGBM uses no fitted preprocessing),
so the main suspect is a **distribution shift** between `application_train` and `application_test`.

**Adversarial validation:** stack train and test, label each row `is_test` (0/1), and ask a classifier to tell
them apart. If it cannot (ROC-AUC ≈ 0.5), the two sets look alike. If it can, its most important features
show *where* they differ.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from src.data import load
from src.features import application_features
from src.viz import set_style, save, BLUE, ORANGE, INK_2, SURFACE

set_style()

In [2]:
train, test = load("application_train"), load("application_test")
print(f"SK_ID_CURR range  train: {train.SK_ID_CURR.min():,} – {train.SK_ID_CURR.max():,}"
      f"   test: {test.SK_ID_CURR.min():,} – {test.SK_ID_CURR.max():,}")

Xa = pd.concat([application_features(train), application_features(test)], ignore_index=True)
for c in Xa.select_dtypes(exclude="number").columns:  # concat of differing categories -> plain text; re-unify
    Xa[c] = Xa[c].astype(str).replace("nan", np.nan).astype("category")
is_test = pd.Series(np.r_[np.zeros(len(train)), np.ones(len(test))].astype(int), name="is_test")
print(f"stacked: {Xa.shape[0]:,} rows ({len(train):,} train + {len(test):,} test) × {Xa.shape[1]} features")

SK_ID_CURR range  train: 100,002 – 456,255   test: 100,001 – 456,250


stacked: 356,255 rows (307,511 train + 48,744 test) × 123 features


In [3]:
def adversarial_auc(X: pd.DataFrame, y: pd.Series, seed: int = 42):
    """5-fold CV ROC-AUC of 'is this row from test?', plus mean gain importance."""
    oof, imp = np.zeros(len(y)), pd.Series(0.0, index=X.columns)
    for tr, va in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y):
        m = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                           random_state=seed, verbose=-1, importance_type="gain")
        m.fit(X.iloc[tr], y.iloc[tr])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        imp += pd.Series(m.feature_importances_, index=X.columns) / 5
    return roc_auc_score(y, oof), (imp / imp.sum()).sort_values(ascending=False), oof

auc_all, imp_all, adv_oof = adversarial_auc(Xa, is_test)
print(f"Adversarial ROC-AUC, all features: {auc_all:.4f}")
imp_all.head(10).map("{:.1%}".format).rename("share of total gain").to_frame()

Adversarial ROC-AUC, all features: 0.8902


,share of total gain
AMT_CREDIT,15.5%
AMT_REQ_CREDIT_BUREAU_QRT,12.8%
AMT_ANNUITY,11.9%
AMT_REQ_CREDIT_BUREAU_MON,10.8%
AMT_GOODS_PRICE,8.2%
NAME_CONTRACT_TYPE,5.7%
CREDIT_TO_INCOME,4.9%
FLAG_EMAIL,4.9%
EXT_SOURCE_1,3.1%
DAYS_LAST_PHONE_CHANGE,2.6%


## Where do train and test differ?
Side-by-side summary of the most "revealing" features.

In [4]:
top = [c for c in imp_all.index[:10] if c in Xa.select_dtypes("number").columns]
tr_part, te_part = Xa.iloc[:len(train)], Xa.iloc[len(train):]
num_cmp = pd.DataFrame({
    "train median": tr_part[top].median(), "test median": te_part[top].median(),
    "train mean": tr_part[top].mean(), "test mean": te_part[top].mean(),
    "train missing": tr_part[top].isna().mean(), "test missing": te_part[top].isna().mean(),
})
num_cmp.style.format({c: "{:,.2f}" for c in num_cmp.columns if "missing" not in c}
                     | {c: "{:.1%}" for c in num_cmp.columns if "missing" in c})

,train median,test median,train mean,test mean,train missing,test missing
AMT_CREDIT,"513,531.00","450,000.00","599,026.00","516,740.44",0.0%,0.0%
AMT_REQ_CREDIT_BUREAU_QRT,0.00,0.00,0.27,0.55,13.5%,12.4%
AMT_ANNUITY,"24,903.00","26,199.00","27,108.57","29,426.24",0.0%,0.0%
AMT_REQ_CREDIT_BUREAU_MON,0.00,0.00,0.27,0.01,13.5%,12.4%
AMT_GOODS_PRICE,"450,000.00","396,000.00","538,396.19","462,618.84",0.1%,0.0%
CREDIT_TO_INCOME,3.27,2.67,3.96,3.17,0.0%,0.0%
FLAG_EMAIL,0.00,0.00,0.06,0.16,0.0%,0.0%
EXT_SOURCE_1,0.51,0.51,0.50,0.50,56.4%,42.1%
DAYS_LAST_PHONE_CHANGE,-757.00,-863.00,-962.86,"-1,077.77",0.0%,0.0%


In [5]:
for col in ["NAME_CONTRACT_TYPE", "FLAG_EMAIL"]:
    print(pd.DataFrame({"train": tr_part[col].value_counts(normalize=True),
                        "test": te_part[col].value_counts(normalize=True)}).map("{:.1%}".format), "\n")

                    train   test
NAME_CONTRACT_TYPE              
Cash loans          90.5%  99.1%
Revolving loans      9.5%   0.9% 

            train   test
FLAG_EMAIL              
0           94.3%  83.7%
1            5.7%  16.3% 



In [6]:
for col in ["AMT_REQ_CREDIT_BUREAU_QRT", "AMT_REQ_CREDIT_BUREAU_MON"]:
    vc = pd.DataFrame({"train": tr_part[col].value_counts(normalize=True, dropna=False),
                       "test": te_part[col].value_counts(normalize=True, dropna=False)}).fillna(0)
    print(col); print(vc.sort_index().head(6).map("{:.1%}".format), "\n")

AMT_REQ_CREDIT_BUREAU_QRT
                           train   test
AMT_REQ_CREDIT_BUREAU_QRT              
0.0                        70.1%  48.3%
1.0                        11.0%  31.9%
2.0                         4.7%   6.2%
3.0                         0.6%   1.0%
4.0                         0.2%   0.1%
5.0                         0.0%   0.0% 

AMT_REQ_CREDIT_BUREAU_MON
                           train   test
AMT_REQ_CREDIT_BUREAU_MON              
0.0                        72.3%  86.9%
1.0                        10.8%   0.7%
2.0                         1.8%   0.0%
3.0                         0.6%   0.0%
4.0                         0.3%   0.0%
5.0                         0.2%   0.0% 



## Does the shift hurt the default model?
The adversarial model gives every **train** client a "test-likeness" score. If the default model is weaker on the
most test-like train clients, the shift explains (part of) the CV → Kaggle gap.

In [7]:
oof_base = pd.read_parquet("../data/processed/oof_baseline.parquet")
assert (oof_base["SK_ID_CURR"].values == train["SK_ID_CURR"].values).all()
likeness = pd.Series(adv_oof[:len(train)], name="test_likeness")
q = pd.qcut(likeness, 5, labels=["1 · least test-like", "2", "3", "4", "5 · most test-like"])
by_q = pd.DataFrame({
    "clients": q.value_counts().sort_index(),
    "mean test-likeness": likeness.groupby(q).mean(),
    "default rate": oof_base["TARGET"].groupby(q).mean(),
    "LightGBM AUC": oof_base.groupby(q).apply(lambda d: roc_auc_score(d["TARGET"], d["LightGBM"])),
    "LogReg (flags) AUC": oof_base.groupby(q).apply(
        lambda d: roc_auc_score(d["TARGET"], d["LogReg (median + missing flags)"])),
})
by_q.style.format({"clients": "{:,}", "mean test-likeness": "{:.3f}", "default rate": "{:.2%}",
                   "LightGBM AUC": "{:.4f}", "LogReg (flags) AUC": "{:.4f}"})

,clients,mean test-likeness,default rate,LightGBM AUC,LogReg (flags) AUC
test_likeness,,,,,
1 · least test-like,"61,503",0.006,6.91%,0.7551,0.7451
2,"61,502",0.018,7.82%,0.7585,0.7506
3,"61,502",0.053,9.24%,0.7624,0.7515
4,"61,502",0.122,8.64%,0.7561,0.7462
5 · most test-like,"61,502",0.307,7.76%,0.7535,0.7422


The test set is 99% cash loans. Is the model simply worse on cash loans?

In [8]:
ctype = train["NAME_CONTRACT_TYPE"].astype(str).values
print(f"{'all train':<18} AUC {roc_auc_score(oof_base.TARGET, oof_base.LightGBM):.4f}")
for k in ["Cash loans", "Revolving loans"]:
    d = oof_base[ctype == k]
    print(f"{k:<18} AUC {roc_auc_score(d.TARGET, d.LightGBM):.4f}  ({len(d):,} clients, default {d.TARGET.mean():.2%})")

all train          AUC 0.7585
Cash loans         AUC 0.7571  (278,232 clients, default 8.35%)
Revolving loans    AUC 0.7573  (29,279 clients, default 5.48%)


**Findings.**
* The shift is strong: a classifier separates train from test with **ROC-AUC 0.89**. IDs are interleaved, so it is
  not simply "test = later customers by ID".
* **Product mix:** revolving loans are 9.5% of train but 0.9% of test; test loans are smaller (median credit 450k vs 514k).
* **Credit-bureau enquiries look recorded differently:** "1 enquiry in the last month" is 10.8% of train but 0.7% of test,
  while "1 enquiry in the last quarter" jumps from 11.0% to 31.9%. The same behaviour seems to land in a different
  column, so a model that learned the train meaning of `AMT_REQ_CREDIT_BUREAU_MON` is misled on test.
* `FLAG_EMAIL` 5.7% → 16.3%; `EXT_SOURCE_1` missing 56% → 42%.
* **Impact inside train is modest:** AUC on the most test-like fifth of train is 0.7535 vs 0.7585 overall, and AUC within
  cash loans alone is 0.7571 (losing the easy cash-vs-revolving separation). These explain part of the 0.013 gap; the rest
  cannot be measured in train because no train client is truly test-like (mean test-likeness of the top fifth: 0.31).
* **Consequence for this project:** model *comparisons* use the same CV for every model and stay valid; absolute Kaggle
  scores will sit below CV. Features whose recording changed between train and test are candidates for removal.